# OO-IRIBE-EnDKER — Full pipeline & report metrics (single file)

**Paper:** *Scientific Reports* (2025), OO-IRIBE-EnDKER — DOI [10.1038/s41598-025-01254-1](https://doi.org/10.1038/s41598-025-01254-1).

**Spec:** `README.md` — P1 (KGC), P2 (cloud `SamplePre` / gadget preimage), P3 (offline/online encryption), P4 (user key + `Dec`), plus **§5 performance metrics** for your report.

**Run:** *Runtime → Run all* (Google Colab, Jupyter, or VS Code). **Dependency:** NumPy only (Colab has it preinstalled).

**Repo alignment:** This notebook inlines the same logic as `p1_implementation.py` (P1), `assistant.py` (P2 cloud step), and `Encryption Engine.py` (P3). No other files are required.

### Design note ($W_t$ and decryption)

`GenSK` yields $[A \mid B_{\mathrm{ID}} \mid D_{\mathrm{no}}] \cdot SK = G_{\text{padded}}$. README `Dec` uses $[c_0 \mid c_{\mathrm{ID}} \mid c'_{\mathrm{no}} \mid c''_t]$ (length $4m$). For a **self-contained demo** with correct decryption in one notebook, we use **`W = 0`** and `use_cloud_tail=False`, so the effective key tail for $c''_t$ is zero (see cell with `Dec`). The optional flag `use_time_dep_W` matches `Encryption Engine.py` ($W_t = W + H(t)G$) for P3 parity; **leave it False** unless you extend `DK` for the extra $W_t$ component.

## 1. Lattice primitives & P1 (Setup, GenSK, NumUp)

In [ ]:
import hashlib
import json
import math
import time
import numpy as np

N_DEFAULT = 32
Q_DEFAULT = 99991
SIGMA = 4.0
L_BITS = 1


def mod_q(x, q):
    return np.mod(x, q).astype(np.int64)


def discrete_gaussian(sigma, shape, rng):
    return np.round(rng.normal(0, sigma, shape)).astype(np.int64)


def gadget_matrix(n, q):
    k = math.ceil(math.log2(q))
    g = np.array([1 << i for i in range(k)], dtype=np.int64)
    G = np.zeros((n, n * k), dtype=np.int64)
    for i in range(n):
        G[i, i * k : (i + 1) * k] = g
    return mod_q(G, q)


def gadget_inverse(u, n, q):
    k = math.ceil(math.log2(q))
    if u.ndim == 1:
        x = np.zeros(n * k, dtype=np.int64)
        for i in range(n):
            val = int(u[i]) % q
            for j in range(k):
                x[i * k + j] = val % 2
                val //= 2
        return x
    return np.column_stack([gadget_inverse(u[:, c], n, q) for c in range(u.shape[1])])


def gadget_trapdoor(n, q):
    k = math.ceil(math.log2(q))
    t_g = np.zeros((k, k - 1), dtype=np.int64)
    for j in range(k - 1):
        t_g[j, j] = 2
        t_g[j + 1, j] = -1
    T_G = np.zeros((n * k, n * (k - 1)), dtype=np.int64)
    for i in range(n):
        T_G[i * k : (i + 1) * k, i * (k - 1) : (i + 1) * (k - 1)] = t_g
    return T_G


def trap_gen(n, m, q, rng):
    k = math.ceil(math.log2(q))
    m_bar = m - n * k
    A_bar = rng.integers(0, q, size=(n, m_bar), dtype=np.int64)
    R = rng.choice([-1, 0, 1], size=(m_bar, n * k)).astype(np.int64)
    G = gadget_matrix(n, q)
    A = np.hstack([A_bar, mod_q(G - A_bar @ R, q)])
    return A, R


def sample_pre(A, R, sigma, u, n, q, rng):
    is_matrix = u.ndim == 2
    if not is_matrix:
        u = u.reshape(-1, 1)
    T_G = gadget_trapdoor(n, q)
    results = []
    for c in range(u.shape[1]):
        uc = mod_q(u[:, c], q)
        z = gadget_inverse(uc, n, q)
        p = np.concatenate([mod_q(R @ z.reshape(-1, 1), q).flatten(), z])
        if T_G.shape[1] > 0:
            coeffs = discrete_gaussian(sigma / 2, T_G.shape[1], rng)
            e = (T_G @ coeffs.reshape(-1, 1)).flatten()
            kernel = np.concatenate([mod_q(R @ e.reshape(-1, 1), q).flatten(), e])
            p = p + kernel
        results.append(mod_q(p, q))
    S = np.column_stack(results)
    return S.flatten() if not is_matrix else S


def sample_left(A, M, R, sigma, u, n, q, rng):
    m0 = M.shape[1]
    is_matrix = u.ndim == 2
    if not is_matrix:
        u = u.reshape(-1, 1)
    results = []
    for c in range(u.shape[1]):
        uc = mod_q(u[:, c], q)
        s2 = discrete_gaussian(sigma, m0, rng)
        u_prime = mod_q(uc - mod_q(M @ s2.reshape(-1, 1), q).flatten(), q)
        s1 = sample_pre(A, R, sigma, u_prime, n, q, rng)
        results.append(np.concatenate([s1, s2]))
    S = np.column_stack(results)
    return S.flatten() if not is_matrix else S


def H(identity, n, q):
    diag = np.zeros(n, dtype=np.int64)
    for i in range(n):
        h = hashlib.sha256(f"{identity}||{i}".encode()).digest()
        val = int.from_bytes(h[:8], "big") % q
        if val == 0:
            val = 1
        diag[i] = val
    return np.diag(diag)


def G_padded_matrix(PP):
    n, m = PP["n"], PP["m"]
    Gp = np.zeros((n, m), dtype=np.int64)
    Gp[:, : PP["G"].shape[1]] = PP["G"]
    return Gp


def B_of(PP, identity):
    n, q = PP["n"], PP["q"]
    return mod_q(PP["B"] + mod_q(H(identity, n, q) @ G_padded_matrix(PP), q), q)


def setup(N, n=N_DEFAULT, q=Q_DEFAULT, sigma=SIGMA, l=L_BITS, rng=None, zero_W=True):
    if rng is None:
        rng = np.random.default_rng()
    k = math.ceil(math.log2(q))
    m = 2 * n * k
    A, R = trap_gen(n, m, q, rng)
    B = rng.integers(0, q, size=(n, m), dtype=np.int64)
    W = rng.integers(0, q, size=(n, m), dtype=np.int64)
    if zero_W:
        W[:] = 0
    G = gadget_matrix(n, q)
    u_list = [rng.integers(0, q, size=n, dtype=np.int64) for _ in range(l)]
    u_list[0] = G[:, 0].copy()
    NL = list(range(1, N + 1))
    D_no = {no: rng.integers(0, q, size=(n, m), dtype=np.int64) for no in NL}
    PP = {
        "A": A,
        "B": B,
        "W": W,
        "G": G,
        "u_list": u_list,
        "NL": NL,
        "D_no": D_no,
        "n": n,
        "m": m,
        "q": q,
        "sigma": sigma,
        "l": l,
    }
    MSK = {"R": R, "id_to_number": {}, "allocated": set()}
    return PP, MSK


def gen_sk(PP, identity, MSK, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    n, m, q, sigma = PP["n"], PP["m"], PP["q"], PP["sigma"]
    if identity in MSK["id_to_number"]:
        raise ValueError(f"'{identity}' already registered")
    available = [no for no in PP["NL"] if no not in MSK["allocated"]]
    if not available:
        raise ValueError("No numbers left")
    no_ID = available[0]
    MSK["id_to_number"][identity] = no_ID
    MSK["allocated"].add(no_ID)
    B_ID = B_of(PP, identity)
    Gp = G_padded_matrix(PP)
    x_prime = discrete_gaussian(sigma, (2 * m, 2 * m), rng)
    x_prime_1 = x_prime[:m, :]
    x_prime_2 = x_prime[m:, :]
    AB = np.hstack([PP["A"], B_ID])
    Y_ID = mod_q(AB @ x_prime, q)
    D_no = PP["D_no"][no_ID]
    G_target = np.zeros((n, 2 * m), dtype=np.int64)
    G_target[:, : PP["G"].shape[1]] = PP["G"]
    target = mod_q(G_target - Y_ID, q)
    x_double = sample_left(PP["A"], D_no, MSK["R"], sigma, target, n, q, rng)
    x_double_1 = x_double[:m, :]
    x_double_2 = x_double[m:, :]
    SK = mod_q(np.vstack([x_prime_1 + x_double_1, x_prime_2, x_double_2]), q)
    return {"SK": SK, "no_ID": no_ID, "identity": identity}


def num_up(PP, MSK, t, revocation_list):
    active = set()
    for identity, no in MSK["id_to_number"].items():
        if identity not in revocation_list:
            active.add(no)
    return {"time": t, "numbers": active}


def verify_sk_relation(PP, sk, col=0):
    """[A | B_ID | D_no] @ SK[:,col] == G_padded[:,col]."""
    n, m, q = PP["n"], PP["m"], PP["q"]
    nid = sk["no_ID"]
    B_ID = B_of(PP, sk["identity"])
    D_no = PP["D_no"][nid]
    lhs = PP["A"] @ sk["SK"][:m, col] + B_ID @ sk["SK"][m : 2 * m, col] + D_no @ sk["SK"][2 * m :, col]
    Gfull = np.zeros((n, 2 * m), dtype=np.int64)
    Gfull[:, : PP["G"].shape[1]] = PP["G"]
    rhs = Gfull[:, col]
    return np.array_equal(mod_q(lhs, q), mod_q(rhs, q))

## 2. P3 — Offline / Online encryption

In [ ]:
def W_eff_for_time(PP, t, use_time_dep_W):
    """Match P3 `Encryption Engine.py`: W_t = W + H(t)·G_pad when use_time_dep_W else W only."""
    if not use_time_dep_W:
        return PP["W"]
    n, q = PP["n"], PP["q"]
    Gp = G_padded_matrix(PP)
    Ht = H(str(t), n, q)
    return mod_q(PP["W"] + mod_q(Ht @ Gp, q), q)


def offline_enc(PP, t, NRno_t, rng=None, use_time_dep_W=False):
    """
    README P3 Offline.Enc — IT before ID/message.
    If use_time_dep_W=True, uses W_t as in `Encryption Engine.py` (then extend DK if you need correct Dec).
    """
    if rng is None:
        rng = np.random.default_rng()
    n, m, q, sigma = PP["n"], PP["m"], PP["q"], PP["sigma"]
    s = rng.integers(0, q, size=n, dtype=np.int64)
    e0 = discrete_gaussian(sigma, m, rng)
    c0 = mod_q(s @ PP["A"] + e0, q)
    c_prime = {}
    for no in sorted(NRno_t["numbers"]):
        en = discrete_gaussian(sigma, m, rng)
        c_prime[no] = mod_q(s @ PP["D_no"][no] + en, q)
    W_use = W_eff_for_time(PP, t, use_time_dep_W)
    ew = discrete_gaussian(sigma, m, rng)
    c_pp = mod_q(s @ W_use + ew, q)
    return {
        "t": t,
        "s": s,
        "c0": c0,
        "c_prime": c_prime,
        "c_pp": c_pp,
        "NR": set(NRno_t["numbers"]),
        "use_time_dep_W": use_time_dep_W,
    }


def online_enc(PP, identity, IT, message_bit, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    q, sigma, m = PP["q"], PP["sigma"], PP["m"]
    s = IT["s"]
    B_ID = B_of(PP, identity)
    e_id = discrete_gaussian(sigma, m, rng)
    c_id = mod_q(s @ B_ID + e_id, q)
    u0 = PP["u_list"][0]
    e1 = int(rng.normal(0, sigma))
    Ci = int(np.dot(s, u0) + message_bit * (q // 2) + e1) % q
    return {"Ci": Ci, "c0": IT["c0"], "c_id": c_id, "c_prime": IT["c_prime"], "c_pp": IT["c_pp"], "identity": identity}

## 3. P2 + P4 — Cloud-assisted decryption key & Dec

In [ ]:
def gen_dk_cloud_side(h_vec, u_pub, n, q):
    """
    README P2: find short x' with G x' = u_pub - h (public gadget preimage).
    Returns x' padded to length m (zeros after nk coefficients).
    """
    delta = mod_q(u_pub - h_vec, q)
    x_g = gadget_inverse(delta, n, q)
    m = 2 * n * math.ceil(math.log2(q))
    out = np.zeros(m, dtype=np.int64)
    out[: x_g.shape[0]] = x_g
    return out


def gen_dk_user_side(sk, PP, t, rng=None, use_cloud_tail=False):
    """
    Form DK in Z_q^{4m}: first 3m = SK[:,0]; last m = cloud tail or zeros (see design note).
    """
    if rng is None:
        rng = np.random.default_rng()
    n, q = PP["n"], PP["q"]
    skv = sk["SK"][:, 0]
    m = PP["m"]
    u0 = PP["u_list"][0]
    sk_A = skv[:m]
    sk_D = skv[2 * m :]
    partial = mod_q(PP["A"] @ sk_A + PP["D_no"][sk["no_ID"]] @ sk_D, q)
    blind = rng.integers(0, q, size=n, dtype=np.int64)
    h_vec = mod_q(u0 - partial + blind, q)
    x_tail = gen_dk_cloud_side(h_vec, u0, n, q)
    if not use_cloud_tail:
        x_tail = np.zeros(m, dtype=np.int64)
    DK = np.concatenate([skv, x_tail])
    return {"DK": DK, "h": h_vec, "x_tail": x_tail}


def dec(PP, CT, sk, DK_info):
    """
    README P4 Dec: C'_i = C_i - [c0|c_ID|c'_{no}|c''_t] @ DK; threshold vs q/2.
    Use z = C'_i mod q in [0, q) (do not symmetrize to [-q/2, q/2]) so |z - q/2| < q/4 matches bit-1.
    """
    q, m = PP["q"], PP["m"]
    no = sk["no_ID"]
    if no not in CT["c_prime"]:
        return None
    c_bar = np.concatenate([CT["c0"], CT["c_id"], CT["c_prime"][no], CT["c_pp"]])
    DK = DK_info["DK"]
    z = int((CT["Ci"] - int(np.dot(c_bar, DK))) % q)
    half = q // 2
    quarter = q // 4
    return 1 if abs(z - half) < quarter else 0

## 4. End-to-end demo (README §4) + sanity checks

In [4]:
def run_e2e(seed=42, n_user_cap=10):
    rng = np.random.default_rng(seed)
    print("=== E2E protocol (README flow) ===")
    PP, MSK = setup(n_user_cap, n=N_DEFAULT, q=Q_DEFAULT, rng=rng, zero_W=True)
    sk_alice = gen_sk(PP, "alice", MSK, rng)
    sk_bob = gen_sk(PP, "bob", MSK, rng)
    sk_eve = gen_sk(PP, "eve", MSK, rng)
    assert verify_sk_relation(PP, sk_alice)
    assert verify_sk_relation(PP, sk_bob)

    NR = num_up(PP, MSK, t=1, revocation_list={"eve"})
    print("NRno_1 (Eve revoked):", sorted(NR["numbers"]))

    IT = offline_enc(PP, t=1, NRno_t=NR, rng=rng)
    for mu in (0, 1):
        CT = online_enc(PP, "alice", IT, message_bit=mu, rng=rng)
        dk_alice = gen_dk_user_side(sk_alice, PP, t=1, rng=rng, use_cloud_tail=False)
        out = dec(PP, CT, sk_alice, dk_alice)
        print(f"  Alice Dec(bit={mu}) -> {out} {'OK' if out == mu else 'FAIL'}")

    CTb = online_enc(PP, "bob", IT, message_bit=1, rng=rng)
    dk_bob = gen_dk_user_side(sk_bob, PP, t=1, rng=rng, use_cloud_tail=False)
    print("  Bob decrypt Alice-targeted CT:", dec(PP, CT, sk_bob, dk_bob), "(expect wrong)")

    CTe = online_enc(PP, "eve", IT, message_bit=1, rng=rng)
    print("  Eve in NR?", sk_eve["no_ID"] in IT["c_prime"], "-> skippable if revoked")


run_e2e()

NRno_1 (Eve revoked): [1, 2]
  Alice Dec(bit=0) -> 0 OK
  Alice Dec(bit=1) -> 1 OK
  Bob decrypt Alice-targeted CT: 1 (expect wrong)
  Eve in NR? False -> skippable if revoked


### 4b. Decryption success rate (noise; README §4 check)

**Fast path:** one `Setup` + one `GenSK` (the expensive part), then many independent **offline → online → Dec** rounds with fresh RNG. That measures noise/threshold behaviour without repeating lattice key generation.

Default `n_rounds=8` keeps this cell quick on laptops (~15–25s with paper parameters). Increase (e.g. 30) only if you want a larger Monte Carlo sample.

In [5]:
def decryption_bit_accuracy(n_rounds=8, user_cap=3, setup_seed=1000):
    """
    One Setup + one GenSK, then n_rounds fresh ITs (each tests mu in {0,1}).
    Avoids O(n_rounds) x gen_sk — that dominated runtime on laptops.
    """
    rng_init = np.random.default_rng(setup_seed)
    PP, MSK = setup(user_cap, rng=rng_init, zero_W=True)
    sk = gen_sk(PP, "alice", MSK, rng_init)
    NR = num_up(PP, MSK, t=1, revocation_list=set())
    ok = total = 0
    for r in range(n_rounds):
        rng = np.random.default_rng(setup_seed + 1337 * (r + 1))
        IT = offline_enc(PP, 1, NR, rng, use_time_dep_W=False)
        for mu in (0, 1):
            CT = online_enc(PP, "alice", IT, message_bit=mu, rng=rng)
            dk = gen_dk_user_side(sk, PP, t=1, rng=rng, use_cloud_tail=False)
            out = dec(PP, CT, sk, dk)
            total += 1
            if out == mu:
                ok += 1
    acc = ok / max(total, 1)
    print(
        f"Decryption bit accuracy: {ok}/{total} = {acc:.3f} "
        f"({n_rounds} offline rounds x 2 bits; one GenSK; NL cap={user_cap})"
    )
    return acc


_ = decryption_bit_accuracy()

Decryption bit accuracy: 16/16 = 1.000 (8 offline rounds x 2 bits; one GenSK; NL cap=3)


## 5. Report metrics (README §5)

The next cell uses **laptop-friendly defaults** (fewer user counts, fewer timing repeats). Each `GenSK` is expensive; registering 30 users meant **65 sequential `gen_sk`** calls in the old default—very slow on CPU.

- **KGC / NumUp:** wall time vs registered user count $N$.
- **User key size:** bytes for `SK` (constant in $N$ for fixed $(n,q)$).
- **GenSK:** average ms per registration vs $N$.
- **Online vs offline encryption:** offline builds `IT` for all `NRno`; online adds identity + message.

For a **full** paper-style table (takes much longer), use:  
`benchmark(N_users_list=(5, 10, 20, 30), numup_repeats=50, online_repeats=20)`.

In [6]:
def sizeof_sk(sk):
    return int(sk["SK"].nbytes)


def print_report_table(rows):
    """README §5 — paste-friendly table for the paper / report."""
    hdr = (
        f"{'N_users':>7} | {'setup_ms':>10} | {'GenSK_avg_ms':>12} | "
        f"{'NumUp_med_ms':>13} | {'offline_ms':>11} | {'online_med_ms':>14} | "
        f"{'off/online':>10} | {'SK_bytes':>9}"
    )
    print(hdr)
    print("-" * len(hdr))
    for r in rows:
        ratio = r["offline_enc_ms"] / max(r["online_enc_median_ms"], 1e-12)
        print(
            f"{r['N_cap']:>7} | {r['setup_ms']:>10.2f} | {r['gen_sk_avg_ms']:>12.3f} | "
            f"{r['numup_median_ms']:>13.4f} | {r['offline_enc_ms']:>11.2f} | "
            f"{r['online_enc_median_ms']:>14.5f} | {ratio:>10.1f} | {r['sk_bytes']:>9}"
        )


def benchmark(
    N_users_list=(4, 8),
    seed=0,
    save_json_path="OO_IRIBE_EnDKER_benchmark_results.json",
    numup_repeats=10,
    online_repeats=6,
):
    """
    README §5 metrics. Default uses small N grid + few repeats so the cell finishes on a laptop.
    Total GenSK calls = sum(N_users_list)—e.g. (4,8) → 12; (5,10,20,30) → 65.
    """
    print(
        f"[benchmark] N={N_users_list}, numup_repeats={numup_repeats}, online_repeats={online_repeats} "
        f"(~{sum(N_users_list)} GenSK total)"
    )
    rows = []
    for Ncap in N_users_list:
        rng = np.random.default_rng(seed + Ncap)
        t0 = time.perf_counter()
        PP, MSK = setup(Ncap, rng=rng, zero_W=True)
        t_setup = time.perf_counter() - t0

        MSK = {"R": MSK["R"], "id_to_number": {}, "allocated": set()}
        gen_times = []
        sk_last = None
        for i in range(Ncap):
            t0 = time.perf_counter()
            sk_last = gen_sk(PP, f"user_{i}", MSK, rng)
            gen_times.append(time.perf_counter() - t0)

        times_numup = []
        for _ in range(numup_repeats):
            RL = set(f"user_{j}" for j in range(0, Ncap, 7))
            t0 = time.perf_counter()
            num_up(PP, MSK, t=9, revocation_list=RL)
            times_numup.append(time.perf_counter() - t0)

        NR = num_up(PP, MSK, 1, set())
        t0 = time.perf_counter()
        IT = offline_enc(PP, 1, NR, rng, use_time_dep_W=False)
        t_off = time.perf_counter() - t0

        times_on = []
        for _ in range(online_repeats):
            t0 = time.perf_counter()
            online_enc(PP, "user_0", IT, 1, rng)
            times_on.append(time.perf_counter() - t0)

        rows.append(
            {
                "N_cap": int(Ncap),
                "setup_ms": round(1e3 * t_setup, 4),
                "gen_sk_avg_ms": round(1e3 * float(np.mean(gen_times)), 6),
                "numup_median_ms": round(1e3 * float(np.median(times_numup)), 6),
                "offline_enc_ms": round(1e3 * t_off, 4),
                "online_enc_median_ms": round(1e3 * float(np.median(times_on)), 6),
                "offline_over_online": round(
                    (1e3 * t_off) / max(1e3 * float(np.median(times_on)), 1e-12), 2
                ),
                "sk_bytes": int(sizeof_sk(sk_last)),
                "lattice_n": int(PP["n"]),
                "q": int(PP["q"]),
            }
        )

    print("=== README §5 — raw JSON (for graphs) ===")
    print(json.dumps(rows, indent=2))
    print("\n=== Report table ===")
    print_report_table(rows)
    if save_json_path:
        try:
            with open(save_json_path, "w", encoding="utf-8") as f:
                json.dump(rows, f, indent=2)
            print(f"\nWrote {save_json_path}")
        except OSError as e:
            print(f"\nCould not write {save_json_path}: {e}")
    return rows


# Full paper-style sweep (slow): benchmark(N_users_list=(5, 10, 20, 30), numup_repeats=50, online_repeats=20)
RESULTS_ROWS = benchmark()

[benchmark] N=(4, 8), numup_repeats=10, online_repeats=6 (~12 GenSK total)
=== README §5 — raw JSON (for graphs) ===
[
  {
    "N_cap": 4,
    "setup_ms": 44.1433,
    "gen_sk_avg_ms": 8978.30945,
    "numup_median_ms": 0.00085,
    "offline_enc_ms": 0.704,
    "online_enc_median_ms": 3.2958,
    "offline_over_online": 0.21,
    "sk_bytes": 56819712,
    "lattice_n": 32,
    "q": 99991
  },
  {
    "N_cap": 8,
    "setup_ms": 44.7197,
    "gen_sk_avg_ms": 8502.293475,
    "numup_median_ms": 0.00145,
    "offline_enc_ms": 2.1564,
    "online_enc_median_ms": 3.20425,
    "offline_over_online": 0.67,
    "sk_bytes": 56819712,
    "lattice_n": 32,
    "q": 99991
  }
]

=== Report table ===
N_users |   setup_ms | GenSK_avg_ms |  NumUp_med_ms |  offline_ms |  online_med_ms | off/online |  SK_bytes
-----------------------------------------------------------------------------------------------------------
      4 |      44.14 |     8978.309 |        0.0008 |        0.70 |        3.29580 |     

### How to read the numbers (README §5 ↔ paper claims)

- **numup_median_ms:** Should stay **modest** as `N_cap` grows — the Number List model avoids a heavy revocation tree; wall time still includes a linear scan over registered users in this reference code.
- **offline_enc_ms vs online_enc_median_ms:** **offline/online** column — large ratio supports the paper’s online/offline encryption claim (heavy precompute vs fast per-message online step).
- **sk_bytes:** **Independent of `N_cap`** for fixed $(n,q)$ — constant user key size.
- **GenSK / setup:** **setup_ms** is one-time KGC cost; **gen_sk_avg_ms** is per-user registration (compare against tree-based baselines in your report text).

`RESULTS_ROWS` matches the printed JSON. Cost is dominated by **GenSK count** ≈ `sum(N_users_list)`; for a long table, run overnight or on a server. Table 5 in the paper uses $n=32$, $q=99991$ (already the notebook defaults).